In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [ ]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['mmorise/World']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

In [ ]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

In [ ]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

In [ ]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [ ]:
# check if it has the right content
dfinf.head(1)

In [ ]:
#mode a means append, so you have all your projects in the same file
yournetid='your_net_id_here'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [2]:
import sys
!{sys.executable} -m pip install -U python-woc pandas matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [3]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd

woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
print("WoC connected!")

WoC connected!


In [4]:
projects = [
    'LFPy/LFPy',
    'numenta/nupic',
    'sem-jl/sem.jl',
    'CardinalKit/CardinalKit',
    'annecori/EpiEstim',
    'yulunzhang/RCAN',
    'hol137/Jupyter-Notebook-Practice-Physical-Constants-Ratios',
    'qupath/qupath',
    'ceedee666/opensap_python_intro',
    'deepmind/acme'
]

list_df_commits = []
for prj in projects:
    prj = prj.lower().replace('/', '_', 2)
    commits = woc.get_values('p2c', prj)
    df = pd.DataFrame(commits, columns=["sha1"])
    df['project'] = prj
    list_df_commits.append(df)
df = pd.concat(list_df_commits)
df.to_csv('df_commits.csv')
print("Total commits:", len(df))
df.head()

Total commits: 35605


,sha1,project
0,000e7e09404882b28e1772239bab9f90168fb575,lfpy_lfpy
1,001d45acb289d070d41dd70e3975d68ca0b337e5,lfpy_lfpy
2,003bcaa802b003465f501c9922e0a91ff7752fa5,lfpy_lfpy
3,00480cfca512135dca4830806426f8be44d5721e,lfpy_lfpy
4,0054091f7d4462d8d81f3dd1129ae0109df91c08,lfpy_lfpy


In [4]:
import time
from tqdm import tqdm

chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []
for chunk in tqdm(chunks):
    res, err = woc.get_values_many('commit.tch', chunk.to_list())
    res = {k: v[0] for k, v in res.items()}
    time.sleep(1)
    if err:
        print('Got Errors', err)
    for commit_sha, commit in res.items():
        commit_data.append({
            'commit': commit_sha,
            'tree': commit[0],
            'parent': list(commit[1]),
            'author': commit[2][0],
            'author_time': int(commit[2][1]),
            'author_tz': commit[2][2],
            'committer': commit[3][0],
            'committer_time': int(commit[3][1]),
            'committer_tz': commit[3][2],
            'message': commit[4],
        })
df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
print("Total commit records retrieved:", len(df_commit_data))
df_commit_data.head(2)

  0%|                                                                                                              | 3/3561 [00:03<1:00:44,  1.02s/it]

Got Errors {'033811f0e3db61b703f83547108bb12a220afab7': 'Key 033811f0e3db61b703f83547108bb12a220afab7 not found in /da5_fast/All.sha1c/commit_3.tch'}


  0%|▍                                                                                                            | 13/3561 [00:13<1:02:22,  1.05s/it]


KeyboardInterrupt: 

In [5]:
df_commit_data = pd.read_csv('df_commit_data.csv')
print("Total commit records loaded:", len(df_commit_data))

Total commit records loaded: 35522


In [6]:
project_stats = df_commit_data.groupby('project').agg(number_of_commits=('commit', 'count'), number_of_authors=('author', 'nunique'), max_time=('author_time', 'max'), min_time=('author_time', 'min')).reset_index()
project_stats

,project,number_of_commits,number_of_authors,max_time,min_time
0,annecori_epiestim,756,36,1749629840,1446815396
1,cardinalkit_cardinalkit,844,47,1762019030,1567376949
2,ceedee666_opensap_python_intro,739,8,1740597559,1629441025
3,deepmind_acme,1523,132,1762067447,1589569078
4,hol137_jupyter-notebook-practice-physical-cons...,107,4,1734793229,1542880171
5,lfpy_lfpy,2316,38,1762947074,1308649102
6,numenta_nupic,21642,556,1671498852,1318109689
7,qupath_qupath,5354,54,1762538018,1472394701
8,sem-jl_sem.jl,2198,19,1762369530,1576597701
9,yulunzhang_rcan,43,7,1744597513,1530634975


In [14]:
projects = [
    'LFPy/LFPy',
    'numenta/nupic',
    'sem-jl/sem.jl',
    'CardinalKit/CardinalKit',
    'annecori/EpiEstim',
    'yulunzhang/RCAN',
    'hol137/Jupyter-Notebook-Practice-Physical-Constants-Ratios',
    'qupath/qupath',
    'ceedee666/opensap_python_intro',
    'deepmind/acme'
]

# Convert project names to the WoC project names used in df_commit_data
woc_projects = [project.lower().replace('/', '_', 1) for project in projects]

# Convert Unix timestamps to dates, then get year-month
df_commit_data['date'] = pd.to_datetime(df_commit_data['author_time'], unit='s')
df_commit_data['month'] = df_commit_data['date'].dt.to_period('M')

# Create a timeseries CSV for each project
for project in woc_projects:

    # Get only commits for this project
    project_data = df_commit_data[df_commit_data['project'] == project]

    # Count commits for each month
    monthly_commits = project_data.groupby('month').size()

    # Create every month from first commit to last commit
    all_months = pd.period_range(
        start=project_data['month'].min(),
        end=project_data['month'].max(),
        freq='M'
    )

    # Add missing months with 0 commits
    monthly_commits = monthly_commits.reindex(all_months, fill_value=0)

    # Create the required two-column DataFrame
    timeseries = pd.DataFrame({
        'Month': monthly_commits.index.astype(str),
        '#Commits': monthly_commits.values
    })

    # Save using semicolon separators
    filename = 'ccoughl3_commits_timeseries_' + project + '.csv'
    timeseries.to_csv(filename, index=False, sep=';')

    print('Saved:', filename)

Saved: ccoughl3_commits_timeseries_lfpy_lfpy.csv
Saved: ccoughl3_commits_timeseries_numenta_nupic.csv
Saved: ccoughl3_commits_timeseries_sem-jl_sem.jl.csv
Saved: ccoughl3_commits_timeseries_cardinalkit_cardinalkit.csv
Saved: ccoughl3_commits_timeseries_annecori_epiestim.csv
Saved: ccoughl3_commits_timeseries_yulunzhang_rcan.csv
Saved: ccoughl3_commits_timeseries_hol137_jupyter-notebook-practice-physical-constants-ratios.csv
Saved: ccoughl3_commits_timeseries_qupath_qupath.csv
Saved: ccoughl3_commits_timeseries_ceedee666_opensap_python_intro.csv
Saved: ccoughl3_commits_timeseries_deepmind_acme.csv


In [16]:
import matplotlib.pyplot as plt

projects = [
    'LFPy/LFPy',
    'numenta/nupic',
    'sem-jl/sem.jl',
    'CardinalKit/CardinalKit',
    'annecori/EpiEstim',
    'yulunzhang/RCAN',
    'hol137/Jupyter-Notebook-Practice-Physical-Constants-Ratios',
    'qupath/qupath',
    'ceedee666/opensap_python_intro',
    'deepmind/acme'
]

# Convert project names to the WoC project names
woc_projects = [project.lower().replace('/', '_', 1) for project in projects]

# Create a plot for each project
for project in woc_projects:

    # Load the monthly CSV
    filename = 'ccoughl3_commits_timeseries_' + project + '.csv'
    df = pd.read_csv(filename, sep=';')

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(df['Month'], df['#Commits'], marker='o', linestyle='-')

    plt.xlabel('Time (YYYY-MM)')
    plt.ylabel('Number of Commits')
    plt.title(project)
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()

    # Save at 300 DPI
    output_filename = 'ccoughl3_timeseries_' + project + '.png'
    plt.savefig(output_filename, dpi=300)
    plt.close()

    print('Saved:', output_filename)

Saved: ccoughl3_timeseries_lfpy_lfpy.png
Saved: ccoughl3_timeseries_numenta_nupic.png
Saved: ccoughl3_timeseries_sem-jl_sem.jl.png
Saved: ccoughl3_timeseries_cardinalkit_cardinalkit.png
Saved: ccoughl3_timeseries_annecori_epiestim.png
Saved: ccoughl3_timeseries_yulunzhang_rcan.png
Saved: ccoughl3_timeseries_hol137_jupyter-notebook-practice-physical-constants-ratios.png
Saved: ccoughl3_timeseries_qupath_qupath.png
Saved: ccoughl3_timeseries_ceedee666_opensap_python_intro.png
Saved: ccoughl3_timeseries_deepmind_acme.png


In [19]:
import pandas as pd

projects = [
    'lfpy_lfpy',
    'numenta_nupic',
    'sem-jl_sem.jl',
    'cardinalkit_cardinalkit',
    'annecori_epiestim',
    'yulunzhang_rcan',
    'hol137_jupyter-notebook-practice-physical-constants-ratios',
    'qupath_qupath',
    'ceedee666_opensap_python_intro',
    'deepmind_acme'
]

results = []

for project in projects:

    filename = 'ccoughl3_commits_timeseries_' + project + '.csv'
    df_monthly = pd.read_csv(filename, sep=';')

    # Find runs of zero-commit months
    zero = df_monthly['#Commits'] == 0
    groups = (zero != zero.shift()).cumsum()

    zero_groups = df_monthly[zero].groupby(groups[zero])

    gaps = []

    for group_number, group in zero_groups:
        gaps.append({
            'start': group['Month'].iloc[0],
            'end': group['Month'].iloc[-1],
            'length': len(group)
        })

    # Find the longest gap
    if len(gaps) > 0:
        longest_gap = max(gaps, key=lambda x: x['length'])

        longest_start = longest_gap['start']
        longest_end = longest_gap['end']
        longest_length = longest_gap['length']

        # Total commits after the longest gap
        after_gap = df_monthly[
            df_monthly['Month'] > longest_end
        ]

        commits_after_gap = after_gap['#Commits'].sum()

    else:
        longest_start = None
        longest_end = None
        longest_length = 0
        commits_after_gap = 0

    # Count gaps that are at least 3 months
    number_of_gaps = sum(
        gap['length'] >= 3
        for gap in gaps
    )

    results.append({
        'project': project,
        'LongestGapStart (YYYY-MM)': longest_start,
        'LongestGapEnd (YYYY-MM)': longest_end,
        'LongestGapLength': longest_length,
        'NumCommitsAfterLastGap': commits_after_gap,
        'NumberOfGapsInTimeline': number_of_gaps
    })

gap_stats = pd.DataFrame(results)

gap_stats

,project,LongestGapStart (YYYY-MM),LongestGapEnd (YYYY-MM),LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline
0,lfpy_lfpy,2024-05,2024-12,8,19,4
1,numenta_nupic,2021-04,2022-11,20,2,3
2,sem-jl_sem.jl,2023-02,2023-05,4,752,2
3,cardinalkit_cardinalkit,2023-11,2024-10,12,24,4
4,annecori_epiestim,2015-12,2016-05,6,754,11
5,yulunzhang_rcan,2024-06,2025-03,10,5,9
6,hol137_jupyter-notebook-practice-physical-cons...,2018-12,2024-11,72,1,1
7,qupath_qupath,2017-01,2017-07,7,5084,2
8,ceedee666_opensap_python_intro,2022-11,2024-12,26,4,2
9,deepmind_acme,2023-12,2023-12,1,60,0
